# w9_cv.ipynb — 5-fold CV of the R60 core recipes (paper main table)

Fold protocol: 814-game universe permuted once (seed 20260711), fold k = test,
fold k+1 = val, other 3 folds = train docs; all inductive exclusions recomputed
per fold. Budget: 600ep towers, ckpt/50, 2-seed checkpoint probes, vsel pick
topped to 10 seeds. A frozen-embedder baseline is written once per fold.

Corpus: same `/workspace/fusion_cache_w9` upload as w9_a100.ipynb (no new data).
Progress: heartbeat prints the tail of every active log every 3 min.
The final cell AUTO-STOPS the pod when the queue drains.

In [ ]:
# w9_cv.ipynb -- constants
import os, subprocess

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_cv_out"

RECIPES = [                      # paper main-table rows (6 x 5 folds = 30 jobs)
    "wcle_cegate2_icetf",        # champion: CE gated on doc samples + I x2
    "wcle_i2ce_icetf",           # gate ablation pair (CE everywhere + I x2)
    "wcle_ce_cetf",              # discriminative baseline (I = 0)
    "wcle_rgate2_icetf",         # RANDOM coverage-matched gate (control)
    "wcle_nodoc_i2ce_icetf",     # zero doc views ("no long text" floor)
    "wcle_ice_icetf",            # dose midpoint (I x1)
]
N_FOLDS = 5
EPOCHS, CKPT_EVERY, CKPT_SEEDS, TOPUP_SEEDS = 600, 50, 2, 10

def _detect_gpus():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=5).stdout.strip()
        ids = [l.strip() for l in out.splitlines() if l.strip()]
        return ids if ids else ["0"]
    except Exception:
        return ["0"]

GPUS = _detect_gpus()
os.makedirs(OUT_DIR, exist_ok=True)
print("jobs :", len(RECIPES) * N_FOLDS, "| gpus:", GPUS)

In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break

In [ ]:
# Stage the corpus into RAM (same file set as w9_a100.ipynb).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz", "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# (recipe x fold) queue across GPUs, with a progress heartbeat.
import os, queue, subprocess, threading, time
from pathlib import Path

def _monitor(log_dir, stop_evt, period=180):
    """Progress heartbeat: every few minutes print the last line of every
    log that changed since the previous beat."""
    seen = {}
    while not stop_evt.wait(period):
        for lg in sorted(log_dir.glob("*.log")):
            try:
                sz = lg.stat().st_size
                if seen.get(lg.name) == sz:
                    continue
                seen[lg.name] = sz
                with open(lg, "rb") as fh:
                    fh.seek(max(0, sz - 400))
                    tail = fh.read().decode(errors="ignore").strip().splitlines()
                if tail:
                    print(f"[beat] {lg.name}: {tail[-1]}", flush=True)
            except Exception:
                pass

jobs = queue.Queue()
n_jobs = 0
for r in RECIPES:
    for k in range(N_FOLDS):
        if (Path(OUT_DIR) / f"ft4var_w9cv_{r}_fold{k}_best.json").exists():
            print(f"[skip] {r}/fold{k} done")
            continue
        jobs.put((r, k)); n_jobs += 1
log_dir = Path(OUT_DIR) / "logs"
log_dir.mkdir(exist_ok=True)
fails = []

def worker(gpu):
    while True:
        try:
            r, k = jobs.get_nowait()
        except queue.Empty:
            return
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu)
        log = log_dir / f"{r}_fold{k}.log"
        print(f"[gpu{gpu}] start {r}/fold{k}", flush=True)
        t0 = time.time()
        with open(log, "w") as fh:
            p = subprocess.run(
                ["python", "-u", os.path.join(REPO, "Pod/w9_cv_worker.py"),
                 "--data-dir", DATA_DIR, "--out-dir", OUT_DIR, "--repo", REPO,
                 "--arm", r, "--fold", str(k), "--n-folds", str(N_FOLDS),
                 "--epochs", str(EPOCHS), "--ckpt-every", str(CKPT_EVERY),
                 "--ckpt-seeds", str(CKPT_SEEDS), "--topup-seeds", str(TOPUP_SEEDS)],
                stdout=fh, stderr=subprocess.STDOUT, env=env)
        if p.returncode != 0:
            fails.append((r, k, str(log)))
        print(f"[gpu{gpu}] " + ("ok" if p.returncode == 0 else "FAIL")
              + f" {r}/fold{k} [{(time.time()-t0)/60:.1f} min]", flush=True)

stop_evt = threading.Event()
mon = threading.Thread(target=_monitor, args=(log_dir, stop_evt), daemon=True)
mon.start()
threads = [threading.Thread(target=worker, args=(g,)) for g in GPUS]
t0 = time.time()
for t in threads: t.start()
for t in threads: t.join()
stop_evt.set()
print(f"CV finished in {(time.time()-t0)/60:.1f} min; {n_jobs} run, {len(fails)} failed")
for r, k, log in fails:
    print("  FAILED:", r, f"fold{k}", "->", log)

In [ ]:
# Aggregate: per-recipe mean +- std across folds (10 seeds each) + frozen rows.
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
print("== frozen-embedder baseline (per fold) ==")
for f in sorted(Path(OUT_DIR).glob("w9cv_frozen_fold*.json")):
    d = json.loads(f.read_text())
    print("  " + f.stem + ": " + " ".join(f"{v}:{d[v]['h1']:.3f}" for v in VORD))
for r in RECIPES:
    per_fold = []
    for k in range(N_FOLDS):
        j = Path(OUT_DIR) / f"ft4var_w9cv_{r}_fold{k}_best.json"
        if not j.exists():
            continue
        runs = json.loads(j.read_text())["per_seed"]
        row = {v: np.mean([x[v]["h1"] for x in runs]) for v in VORD}
        row["m4"] = np.mean([np.mean([x[v]["h1"] for x in runs]) for v in VORD])
        row["tagn"] = np.mean([x["noname"]["tag"] for x in runs])
        per_fold.append(row)
    if not per_fold:
        continue
    print(f"\n{r} ({len(per_fold)} folds)")
    for kf in ("neutral", "noname", "m4", "tagn"):
        vals = [pf[kf] for pf in per_fold]
        print(f"  {kf:8s} {np.mean(vals):.3f} +- {np.std(vals):.3f}")

In [ ]:
# AUTO-STOP: stop THIS pod when the queue has finished (results live on the
# network volume; idle GPU time is pure waste). Uses the hardened ladder in
# VICReg_review/pod_selfstop.py. Set AUTO_STOP=False to keep the pod alive.
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    if fails:
        print(f"NOTE: {len(fails)} job(s) FAILED -- logs in {OUT_DIR}/logs; "
              "stopping anyway to avoid idle burn.")
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- remember to stop the pod yourself.")